# Chương 1 – Nhập môn Xử lý ảnh
## Bài tập thực hành

Bài thực hành này giúp các bạn:
- Làm quen với môi trường Python cho xử lý ảnh: `os`, `numpy`, `opencv-python` (`cv2`), `matplotlib`.
- Hiểu ảnh số được biểu diễn như thế nào (ảnh số là hàm 2 chiều f(x, y)).
- Thực hành minh họa **3 mức xử lý ảnh số** đã học: xử lý mức thấp, mức trung, mức cao.

**Dữ liệu ảnh**: các ảnh mẫu nằm trong thư mục `images/` cùng cấp với notebook này.

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

IMG_DIR = 'images'

## Bài tập 1: Làm quen với thư viện `os`

Trong các project về xử lý ảnh, ta thường xuyên cần liệt kê, duyệt qua các file ảnh trong một thư mục dữ liệu.
Một số hàm hay dùng:
- `os.listdir(dir)`: liệt kê tên tất cả file/thư mục con trong `dir`.
- `os.path.join(a, b)`: nối đường dẫn đúng theo định dạng hệ điều hành.
- `os.path.abspath(path)`: trả về đường dẫn tuyệt đối.

**Yêu cầu**: Hãy tạo 2 danh sách (python list) `abs_paths` và `rel_paths` chứa đường dẫn tuyệt đối và
đường dẫn tương đối đến tất cả các ảnh nằm trong thư mục `images/`.

In [ ]:
abs_paths = []
rel_paths = []

for name in sorted(os.listdir(IMG_DIR)):
    rel_path = os.path.join(IMG_DIR, name)
    extension = os.path.splitext(name)[1].lower()
    if os.path.isfile(rel_path) and extension in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}:
        rel_paths.append(rel_path)
        abs_paths.append(os.path.abspath(rel_path))

print("Số lượng ảnh tìm được:", len(rel_paths))
print(rel_paths)

## Bài tập 2: Ảnh số là gì? Đọc và biểu diễn ảnh dưới dạng ma trận

Theo lý thuyết Chương 1, một bức ảnh được định nghĩa là một hàm 2 chiều f(x, y), trong đó x, y là toạ độ
không gian và giá trị f(x, y) là cường độ sáng (hoặc màu) tại điểm đó. Khi số hoá, ảnh trở thành một **ma trận
số** mà `numpy`/`OpenCV` có thể xử lý.

- `cv2.imread(path)` đọc ảnh, trả về `numpy.ndarray` 3 chiều `(height, width, channels)`.
- `img.shape`, `img.dtype` cho biết kích thước và kiểu dữ liệu của ảnh (thường là `uint8`, giá trị 0-255).

In [ ]:
img = cv2.imread(os.path.join(IMG_DIR, '1.jpg'))
print("Kiểu dữ liệu trả về:", type(img))
print("Shape (H, W, C):", img.shape)
print("dtype:", img.dtype)

**Yêu cầu**: In ra giá trị pixel tại toạ độ (0, 0) (góc trên-trái) và tại toạ độ chính giữa ảnh. Giải thích ý nghĩa 3 giá trị số bạn nhận được tại mỗi điểm ảnh.

In [ ]:
top_left_pixel = img[0, 0]
center_pixel = img[img.shape[0] // 2, img.shape[1] // 2]

print("Pixel (0,0):", top_left_pixel)
print("Pixel giữa ảnh:", center_pixel)

### Đọc ảnh xám và so sánh với ảnh màu

`cv2.imread(path, cv2.IMREAD_GRAYSCALE)` đọc ảnh trực tiếp ở dạng ảnh xám (1 kênh).

**Yêu cầu**: đọc lại đúng file ảnh trên ở chế độ ảnh xám, in ra `shape` và so sánh với `shape` của ảnh màu
ở trên. Viết 1-2 câu nhận xét (dùng comment `#`) về sự khác biệt.

In [ ]:
gray = cv2.imread(os.path.join(IMG_DIR, '1.jpg'), cv2.IMREAD_GRAYSCALE)
if gray is None:
    raise FileNotFoundError('Không đọc được images/1.jpg')

print("Shape ảnh xám:", gray.shape)
print("Shape ảnh màu:", img.shape)
# Nhận xét: ảnh xám là ma trận 2 chiều (H, W), còn ảnh màu là mảng 3 chiều
# (H, W, 3). Ba giá trị tại mỗi pixel màu lần lượt là cường độ B, G, R.

## Hiển thị ảnh đúng màu (BGR vs RGB)

`OpenCV` lưu ảnh theo thứ tự kênh màu **B, G, R**, trong khi `matplotlib` hiển thị ảnh theo thứ tự **R, G, B**.
Hãy quan sát sự khác biệt:

In [ ]:
plt.figure(figsize=(6, 6))
plt.title('Hiển thị trực tiếp bằng plt.imshow (SAI MÀU)')
plt.imshow(img)
plt.show()

**Yêu cầu**: dùng `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` để chuyển đổi kênh màu rồi hiển thị lại cho đúng màu thật.

In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.title('Sau khi chuyển BGR -> RGB')
plt.imshow(img_rgb)
plt.show()

## Bài tập 3: Minh họa 3 mức xử lý ảnh số

Theo lý thuyết Chương 1, xử lý ảnh số được chia thành 3 mức:
- **Mức thấp** (Low-level): đầu vào và đầu ra đều là ảnh. Ví dụ: loại bỏ nhiễu, tăng độ tương phản, làm
  sắc nét ảnh.
- **Mức trung** (Mid-level): đầu vào là ảnh, đầu ra là các thuộc tính/đặc trưng được trích xuất từ ảnh
  (ví dụ: đường biên, vùng đối tượng sau khi phân ngưỡng).
- **Mức cao** (High-level): đầu vào là các đặc trưng, đầu ra là ý nghĩa/nhận thức (ví dụ: nhận dạng đối
  tượng). Ở mức này thường cần các mô hình phức tạp hơn (sẽ học ở các chương sau).

Dưới đây ta sẽ thực hành minh hoạ mức thấp và mức trung bằng các thao tác xử lý ảnh đơn giản.

### 3.1. Minh họa xử lý mức thấp: khử nhiễu ảnh (ảnh -> ảnh)

Ảnh `deer_salt.jpg` bị nhiễu muối-tiêu (salt-and-pepper noise).

In [ ]:
noisy = cv2.imread(os.path.join(IMG_DIR, 'deer_salt.jpg'), cv2.IMREAD_GRAYSCALE)
plt.figure(figsize=(6, 6))
plt.title('Ảnh gốc bị nhiễu')
plt.imshow(noisy, cmap='gray')
plt.show()

**Yêu cầu**: dùng `cv2.medianBlur(img, ksize)` để khử nhiễu, hiển thị ảnh trước và sau khi lọc trên cùng 1 hình để so sánh. Hãy thử với `ksize = 3` và `ksize = 5`, chọn kết quả bạn thấy tốt hơn.

In [ ]:
# Kernel 5x5 loại nhiễu muối-tiêu tốt hơn 3x3 trên ảnh mẫu này.
denoised = cv2.medianBlur(noisy, 5)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Trước'); plt.imshow(noisy, cmap='gray')
plt.subplot(1, 2, 2); plt.title('Sau khi khử nhiễu'); plt.imshow(denoised, cmap='gray')
plt.show()

### 3.2. Minh họa xử lý mức trung: phân ngưỡng (thresholding) đơn giản (ảnh -> vùng đối tượng)

Ảnh `statue_bright.jpg` có phần bức tượng sáng hơn phần nền.

In [ ]:
statue = cv2.imread(os.path.join(IMG_DIR, 'statue_bright.jpg'), cv2.IMREAD_GRAYSCALE)
plt.figure(figsize=(6, 6))
plt.imshow(statue, cmap='gray')
plt.show()

**Yêu cầu**: viết đoạn code phân ngưỡng thủ công: với ngưỡng `T = 128`, gán pixel có giá trị > T thành 255, ngược lại thành 0 (dùng `numpy`, không dùng hàm có sẵn `cv2.threshold`). Kết quả là một ảnh nhị phân tách vùng "đối tượng" khỏi "nền" — đây là một dạng đặc trưng (vùng ảnh) được trích ra từ ảnh, tương ứng với mức xử lý trung.

In [ ]:
T = 128

binary = np.where(statue > T, 255, 0).astype(np.uint8)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh xám gốc'); plt.imshow(statue, cmap='gray')
plt.subplot(1, 2, 2); plt.title(f'Phân ngưỡng T={T}'); plt.imshow(binary, cmap='gray')
plt.show()

## Bài tập 4 (mở rộng): Hàm hiển thị nhiều ảnh

**Yêu cầu**: viết một hàm tiện ích `show_images(images, titles, cmap=None)` nhận vào danh sách ảnh và
danh sách tên tương ứng, hiển thị tất cả trên cùng một hàng bằng `plt.subplot`. Sau đó dùng lại hàm này
để hiển thị 5 ảnh `1.jpg` đến `5.jpg` trong thư mục `images/`.

In [ ]:
def show_images(images, titles, cmap=None):
    if len(images) != len(titles):
        raise ValueError('Số ảnh phải bằng số tiêu đề')
    if not images:
        return
    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    axes = np.atleast_1d(axes)
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

imgs = [cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, f'{i}.jpg')), cv2.COLOR_BGR2RGB) for i in range(1, 6)]
titles = [f'{i}.jpg' for i in range(1, 6)]
show_images(imgs, titles)